# Hospital Readmission Prediction

Predict 30-day readmission (`readmitted == <30`) using L2-regularized Logistic Regression on the UCI Diabetes 130-US Hospitals dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

DATA_PATH = 'data/diabetic_data.csv'
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df = df.replace('?', np.nan)
df['target_30day'] = (df['readmitted'] == '<30').astype(int)
drop_cols = [c for c in ['encounter_id','patient_nbr','readmitted'] if c in df.columns]
X = df.drop(columns=drop_cols + ['target_30day'])
y = df['target_30day']

# Remove columns with more than 90% missing values.
high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)
print('Target rate:', y.mean())
print('Features:', X.shape[1])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric), ('cat', categorical_pipe, categorical)])
model = Pipeline([('preprocess', preprocess), ('logreg', LogisticRegression(penalty='l2', C=1.0, max_iter=1000, solver='liblinear'))])
model.fit(X_train, y_train)

In [ ]:
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.50).astype(int)
auc = roc_auc_score(y_test, proba)
print(f'ROC-AUC: {auc:.4f}')
print(classification_report(y_test, pred, digits=4))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

In [ ]:
RocCurveDisplay.from_predictions(y_test, proba)
plt.title('ROC Curve - 30-Day Readmission')
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title('Confusion Matrix')
plt.show()

## False-negative vs false-positive cost

A **false negative** is a patient who is readmitted within 30 days but is not flagged. In an operational setting, this can mean a missed opportunity for additional follow-up. A **false positive** is a patient flagged by the model who is not readmitted; repeated false positives can consume care-team time and resources. The appropriate threshold therefore depends on the relative cost assigned to these errors.

This notebook is an educational case study and is not a clinical decision-support system.